# 14 Case Study — Solution

The complete solution for the Songbai Nursing Home Legionnaires' disease mini outbreak investigation report.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || True
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy import stats

# -- CJK font setup (prevents CJK labels from rendering as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150


## Question 1: Outbreak summary table

In [ ]:
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

n_total = len(df)
n_infected = int(df["infected"].sum())
n_deaths = int((df["outcome"] == "dead").sum())
n_hosp = int(df["hospitalized"].sum())
n_icu = int(df["icu_admission"].sum())

summary = pd.DataFrame([
    ["Total residents", n_total, ""],
    ["Infected", n_infected, f"{n_infected/n_total:.1%}"],
    ["Deaths", n_deaths, f"{n_deaths/n_infected:.1%} (CFR)"],
    ["Hospitalized", n_hosp, f"{n_hosp/n_infected:.1%} (hosp. rate)"],
    ["ICU", n_icu, f"{n_icu/n_hosp:.1%} (ICU/hosp.)"],
    ["Attack rate", f"{n_infected/n_total:.1%}", ""],
    ["Case fatality rate", f"{n_deaths/n_infected:.1%}", ""],
], columns=["Metric", "Value", "Proportion"])

print("=== Outbreak summary table ===")
print(summary.to_string(index=False))

## Question 2: Quick risk-factor screening

In [ ]:
factors = ["shower_use", "hydrotherapy_use", "comorbidity_copd", "immunosuppressed"]
results = []

for factor in factors:
    exposed_inf = int(df[(df[factor] == 1) & (df["infected"] == 1)].shape[0])
    exposed_n = int(df[df[factor] == 1].shape[0])
    unexposed_inf = int(df[(df[factor] == 0) & (df["infected"] == 1)].shape[0])
    unexposed_n = int(df[df[factor] == 0].shape[0])

    ar_exp = exposed_inf / exposed_n if exposed_n > 0 else 0
    ar_unexp = unexposed_inf / unexposed_n if unexposed_n > 0 else 0
    rr = ar_exp / ar_unexp if ar_unexp > 0 else float("inf")

    chi2, p, _, _ = stats.chi2_contingency(
        pd.crosstab(df[factor], df["infected"])
    )

    results.append({
        "factor": factor,
        "exposed_AR": f"{ar_exp:.1%}",
        "unexposed_AR": f"{ar_unexp:.1%}",
        "RR": f"{rr:.2f}",
        "p-value": f"{p:.4f}",
        "sig": "*" if p < 0.05 else "",
    })

rr_df = pd.DataFrame(results)
print("=== Risk-factor RR comparison table ===")
print(rr_df.to_string(index=False))

max_rr = rr_df.loc[rr_df["RR"].astype(float).idxmax()]
print(f"\n→ Factor with the largest RR: {max_rr['factor']} (RR = {max_rr['RR']})")
print("→ Shower use is the strongest exposure risk factor and is significantly associated with infection status")

## Question 3 (Challenge): Mini SitRep

In [ ]:
cases = df[df["infected"] == 1].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Chart 1: Epidemic curve ---
daily = cases.groupby("symptom_onset_date").size()
full_range = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
daily = daily.reindex(full_range, fill_value=0)

axes[0].bar(daily.index, daily.values, color="steelblue", edgecolor="white")
peak = daily.idxmax()
axes[0].axvline(peak, color="red", linestyle="--", alpha=0.7)
axes[0].set_title(f"Epidemic curve (peak: {peak.strftime('%m/%d')})")
axes[0].set_ylabel("Daily new cases")
axes[0].tick_params(axis="x", rotation=45)

# --- Chart 2: Age distribution ---
for label, grp in df.groupby("infected"):
    tag = "Infected" if label == 1 else "Not infected"
    axes[1].hist(grp["age"], bins=15, alpha=0.6, label=tag, edgecolor="white")
axes[1].set_title("Age distribution")
axes[1].set_xlabel("Age")
axes[1].legend()

# --- Chart 3: Attack rate by floor and wing ---
zone = df.groupby(["floor", "wing"])["infected"].agg(["sum", "count"]).reset_index()
zone["ar"] = zone["sum"] / zone["count"] * 100
zone["label"] = zone["floor"].astype(str) + "F-" + zone["wing"]
colors = ["#e74c3c" if ar > 50 else "steelblue" for ar in zone["ar"]]
axes[2].bar(zone["label"], zone["ar"], color=colors)
axes[2].axhline(50, color="red", linestyle="--", alpha=0.5)
axes[2].set_title("Attack Rate by Floor and Wing")
axes[2].set_ylabel("Attack Rate (%)")
for i, row in zone.iterrows():
    axes[2].text(i, row["ar"] + 1, f"{row['ar']:.0f}%", ha="center", fontsize=9)

plt.suptitle("Songbai Nursing Home Legionnaires' Disease Outbreak — Mini SitRep", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Action recommendations
top_zone = zone.sort_values("ar", ascending=False).iloc[0]
print("=" * 40)
print("  Action recommendations")
print("=" * 40)
print(f"  1. Priority zone to address: {top_zone['label']} (attack rate {top_zone['ar']:.1f}%)")
print(f"  2. Secondary focus: 2F-A (54.5%) — these two zones account for the majority of cases")
print(f"  3. Immediately shut down showers in the high-risk zones")
print(f"  4. Conduct environmental sampling of the 2F and 3F plumbing systems")
print("=" * 40)

### Interpretation

- **Question 1**: The summary table is the first page of an outbreak report, letting decision-makers quickly grasp the scale
- **Question 2**: RR screening quickly identifies the exposure factors most worth investigating in depth
  - Note: the crude RR is not adjusted for confounders; pair it with stratified analysis (Ch05) and logistic regression (Ch06)
- **Question 3**: A good SitRep must include "action recommendations" -- the purpose of analysis is to support decisions

Congratulations on finishing the final exercise! You now have the core skills to conduct an outbreak investigation with Python.